# 📦 E-Commerce Demand Forecasting Pipeline
**Dataset:** Shopee Sales Data — Dec 2023 to Nov 2025
**Goal:** Predict daily order volume to optimize inventory & fulfillment decisions
**Stack:** Python · Pandas · Seaborn · ARIMA · Prophet · LSTM

---
## Pipeline Overview
```
Raw CSV → Cleaning → Feature Engineering → EDA → Modeling → Evaluation
```

> **Catatan restrukturisasi:** notebook ini dibuat tahan-banting terhadap dependency yang belum
> terinstall. Jika `statsmodels`, `prophet`, atau `tensorflow` tidak tersedia di environment-mu,
> bagian model terkait otomatis di-skip (dengan pesan jelas) tanpa menghentikan seluruh notebook.


## 0. Setup & Imports

In [ ]:
# Install dependencies kalau belum ada (uncomment baris di bawah lalu jalankan sekali)
# !pip install prophet scikit-learn statsmodels tensorflow matplotlib seaborn nbformat

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------------
# Optional heavy dependencies — pipeline tetap jalan meski salah satu
# belum terinstall; bagian model terkait akan di-skip otomatis.
# ------------------------------------------------------------------
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print('⚠️  statsmodels tidak ditemukan -> bagian ADF test & ARIMA akan dilewati.')
    print('    Install dengan: pip install statsmodels')

try:
    from prophet import Prophet
    HAS_PROPHET = True
except ImportError:
    HAS_PROPHET = False
    print('⚠️  prophet tidak ditemukan -> bagian model Prophet akan dilewati.')
    print('    Install dengan: pip install prophet')

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print('⚠️  tensorflow tidak ditemukan -> bagian model LSTM akan dilewati.')
    print('    Install dengan: pip install tensorflow')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
PALETTE = 'Blues_d'

print('\n✅ Imports selesai')
if HAS_TF:
    print(f'TensorFlow version: {tf.__version__}')


---
## 1. Data Loading & Initial Inspection

In [ ]:
# Load raw dataset
import os

FILE_PATH = 'all_months_clean.csv'   # taruh CSV ini di folder yang sama dengan notebook

if not os.path.exists(FILE_PATH):
    # fallback umum kalau dijalankan dari direktori lain
    for candidate in ['/mnt/user-data/uploads/all_months_clean.csv',
                       './data/all_months_clean.csv']:
        if os.path.exists(candidate):
            FILE_PATH = candidate
            break
    else:
        raise FileNotFoundError(
            f"File '{FILE_PATH}' tidak ditemukan. Pastikan all_months_clean.csv "
            "berada di folder yang sama dengan notebook ini, atau ubah FILE_PATH di atas."
        )

df_raw = pd.read_csv(
    FILE_PATH,
    sep=';',
    encoding='utf-8-sig'
)

print(f'Shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head(3)


In [ ]:
# Quick data health check
print('=== NULL VALUES ===')
print(df_raw.isnull().sum())

print('\n=== DATA TYPES ===')
print(df_raw.dtypes)

print('\n=== STATUS PESANAN VALUE COUNTS ===')
print(df_raw['Status Pesanan'].value_counts())


---
## 2. Data Cleaning

**Issues identified:**
- `Waktu Pesanan Dibuat`: masih string, perlu konversi datetime — ada baris dengan nilai null
- `Alasan Pembatalan`: null untuk order yang tidak dibatalkan (wajar, isi dengan 'No Cancellation')
- `Status Pesanan`: string panjang & tidak konsisten → perlu normalisasi
- `Opsi Pengiriman`: 45+ varian → perlu dikonsolidasi jadi 5 kategori bersih
- `Total Pembayaran`: ada baris bernilai 0 (semua order yang dibatalkan)


In [ ]:
df = df_raw.copy()

# ------------------------------------------------------------------
# 2.1  Rename columns -> snake_case English untuk konsistensi downstream
#      (urutan kolom HARUS sama dengan urutan asli di CSV)
# ------------------------------------------------------------------
expected_n_cols = 19
assert df.shape[1] == expected_n_cols, (
    f"Jumlah kolom CSV ({df.shape[1]}) tidak sama dengan yang diharapkan "
    f"({expected_n_cols}). Cek ulang struktur file sumber sebelum rename."
)

df.columns = [
    'order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty',
    'total_discount', 'product_categories', 'num_product_categories',
    'order_status_raw', 'cancel_reason', 'shipping_option',
    'payment_method', 'city', 'province',
    'shipping_paid_by_buyer', 'shipping_discount',
    'total_revenue', 'estimated_shipping_cost',
    'order_time_raw', 'source_file'
]

# ------------------------------------------------------------------
# 2.2  Parse datetime — coerce errors menghasilkan NaT untuk nilai rusak
# ------------------------------------------------------------------
df['order_time'] = pd.to_datetime(df['order_time_raw'], errors='coerce')

before_drop = len(df)
df = df.dropna(subset=['order_time'])       # drop baris tanpa timestamp valid
print(f'Dropped {before_drop - len(df)} rows with null timestamps -> {len(df)} rows remaining')

# ------------------------------------------------------------------
# 2.3  Normalize order status -> kategori bersih
# ------------------------------------------------------------------
def normalize_status(s: str) -> str:
    if s == 'Selesai':                           return 'Completed'
    if s == 'Batal':                              return 'Cancelled'
    if s.startswith('Pesanan diterima'):          return 'Delivered'
    if s in ('Sedang Dikirim', 'Telah Dikirim'):  return 'In Transit'
    return 'Other'

df['order_status'] = df['order_status_raw'].apply(normalize_status)
print('\nStatus distribution after normalization:')
print(df['order_status'].value_counts())

# ------------------------------------------------------------------
# 2.4  Fill non-cancelled reason dengan placeholder
# ------------------------------------------------------------------
df['cancel_reason'] = df['cancel_reason'].fillna('No Cancellation')

# ------------------------------------------------------------------
# 2.5  Konsolidasi 45+ varian opsi pengiriman -> 5 kategori bersih
# ------------------------------------------------------------------
def consolidate_shipping(s: str) -> str:
    s = str(s).lower()
    if 'hemat' in s:                          return 'Economy'
    if 'same day' in s or 'sameday' in s:      return 'Same Day'
    if 'instant' in s:                         return 'Instant'
    if 'next day' in s or 'yes' in s:          return 'Next Day'
    return 'Regular'

df['shipping_type'] = df['shipping_option'].apply(consolidate_shipping)
print('\nShipping type distribution:')
print(df['shipping_type'].value_counts())

# ------------------------------------------------------------------
# 2.6  Remove duplicate order IDs (keep first occurrence)
# ------------------------------------------------------------------
dupes_before = df.duplicated('order_id').sum()
df = df.drop_duplicates('order_id', keep='first')
print(f'\nRemoved {dupes_before} duplicate order IDs')

print(f'\n✅ Clean dataset shape: {df.shape}')


In [ ]:
# Quick sanity check after cleaning
print('Remaining nulls:')
print(df[['order_time','order_status','shipping_type','total_revenue']].isnull().sum())
print(f'\nDate range: {df["order_time"].min().date()} -> {df["order_time"].max().date()}')
df.head(3)


---
## 3. Feature Engineering

Membangun 18 fitur dari timestamp mentah dan kolom yang sudah ada.
Fitur-fitur ini melayani **dua tujuan**:
1. EDA — memahami demand driver
2. Modeling — regressor untuk Prophet, input sequence untuk LSTM


In [ ]:
# ------------------------------------------------------------------
# 3.1  Temporal features dari timestamp
# ------------------------------------------------------------------
df['date']         = df['order_time'].dt.date
df['year']         = df['order_time'].dt.year
df['month']        = df['order_time'].dt.month
df['day_of_month'] = df['order_time'].dt.day
df['day_of_week']  = df['order_time'].dt.dayofweek      # 0=Mon, 6=Sun
df['week_of_year'] = df['order_time'].dt.isocalendar().week.astype(int)
df['hour']         = df['order_time'].dt.hour
df['quarter']      = df['order_time'].dt.quarter

# ------------------------------------------------------------------
# 3.2  Business-logic flags
# ------------------------------------------------------------------
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Shopee promo date proxy (flash sale tanggal 1, 10, 15, 25 tiap bulan)
df['is_promo_date'] = df['day_of_month'].isin([1, 10, 15, 25]).astype(int)

# Double-date sale proxy (1/1, 2/2 ... 11/11, 12/12)
df['is_double_date'] = (df['month'] == df['day_of_month']).astype(int)

# Payday proxy (akhir bulan: 25-31 = gajian -> demand naik)
df['is_payday_period'] = (df['day_of_month'] >= 25).astype(int)

# Indonesian national holiday flag (manual, kalender 2024-2025)
holidays_id = [
    '2024-01-01','2024-02-08','2024-02-09','2024-02-10',
    '2024-03-11','2024-03-29','2024-04-08','2024-04-09',
    '2024-04-10','2024-04-11','2024-04-12','2024-05-01',
    '2024-05-09','2024-05-23','2024-06-01','2024-06-17',
    '2024-06-18','2024-07-07','2024-08-17','2024-09-16',
    '2024-12-25','2024-12-26',
    '2025-01-01','2025-01-27','2025-01-28','2025-01-29',
    '2025-03-28','2025-03-29','2025-03-30','2025-03-31',
    '2025-04-01','2025-04-18','2025-05-01','2025-05-12',
    '2025-05-29','2025-06-01','2025-06-06','2025-08-17',
    '2025-09-05','2025-12-25'
]
holidays_set = set(pd.to_datetime(holidays_id).strftime('%Y-%m-%d'))
df['is_holiday'] = df['order_time'].dt.strftime('%Y-%m-%d').isin(holidays_set).astype(int)

# Time of day bucket
df['time_of_day'] = pd.cut(
    df['hour'],
    bins=[-1, 5, 11, 17, 22, 24],
    labels=['Night', 'Morning', 'Afternoon', 'Evening', 'Late Night']
)

# ------------------------------------------------------------------
# 3.3  Transaction-level derived features
# ------------------------------------------------------------------
df['revenue_per_item'] = (df['total_revenue'] / df['total_qty'].replace(0, np.nan)).round(0)

df['discount_ratio'] = (
    df['total_discount'] / (df['total_revenue'] + df['total_discount']).replace(0, np.nan)
).round(4)

df['is_cod'] = (df['payment_method'] == 'COD (Bayar di Tempat)').astype(int)

df['shipping_cost_ratio'] = (
    df['estimated_shipping_cost'] / df['total_revenue'].replace(0, np.nan)
).round(4)

print(f'✅ Feature engineering complete — {df.shape[1]} total columns')
new_cols = ['date','year','month','day_of_month','day_of_week','week_of_year',
            'hour','quarter','is_weekend','is_promo_date','is_double_date',
            'is_payday_period','is_holiday','time_of_day',
            'revenue_per_item','discount_ratio','is_cod','shipping_cost_ratio']
print('\nNew columns added:')
print(new_cols)


In [ ]:
# Preview enriched dataset
cols_to_show = ['order_id','order_time','order_status','total_revenue',
                'day_of_week','hour','is_weekend','is_holiday',
                'is_promo_date','is_cod','shipping_type']
df[cols_to_show].head(5)


In [ ]:
# ------------------------------------------------------------------
# 3.4  Build daily time series — target variable utama
# ------------------------------------------------------------------
completed = df[df['order_status'] == 'Completed'].copy()

daily_ts = (
    completed
    .groupby('date')
    .agg(
        order_count        = ('order_id',     'count'),
        total_units        = ('total_qty',    'sum'),
        total_revenue      = ('total_revenue','sum'),
        avg_order_value    = ('total_revenue','mean'),
        holiday_flag       = ('is_holiday',   'max'),
        promo_flag         = ('is_promo_date','max'),
        cod_ratio          = ('is_cod',       'mean'),
    )
    .reset_index()
)

daily_ts['date'] = pd.to_datetime(daily_ts['date'])

# Isi tanggal yang hilang dengan 0 (Prophet & ARIMA butuh series kontinu)
full_range = pd.date_range(daily_ts['date'].min(), daily_ts['date'].max(), freq='D')
daily_ts   = daily_ts.set_index('date').reindex(full_range, fill_value=0).reset_index()
daily_ts.rename(columns={'index':'date'}, inplace=True)

print(f'Daily time series: {len(daily_ts)} rows')
print(f'Date range: {daily_ts["date"].min().date()} -> {daily_ts["date"].max().date()}')
print(f'\nDemand stats:')
print(daily_ts['order_count'].describe().round(2))
daily_ts.head(5)


---
## 4. Exploratory Data Analysis (EDA)

Tujuan: temukan seasonality pattern sebelum pilih model
**4 visualisasi utama:**
1. Daily demand time series + rolling average
2. Heatmap demand: Hour × Day of Week
3. Monthly demand bar chart (seasonality bulanan)
4. Revenue by top category


In [ ]:
# -- 4.1  Daily demand time series --------------------------------
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(daily_ts['date'], daily_ts['order_count'],
        color='#2196F3', linewidth=0.8, alpha=0.6, label='Daily Orders')

ax.plot(daily_ts['date'], daily_ts['order_count'].rolling(7).mean(),
        color='#FF5722', linewidth=1.8, label='7-Day MA')
ax.plot(daily_ts['date'], daily_ts['order_count'].rolling(30).mean(),
        color='#4CAF50', linewidth=1.8, linestyle='--', label='30-Day MA')

mean_d  = daily_ts['order_count'].mean()
std_d   = daily_ts['order_count'].std()
spikes  = daily_ts[daily_ts['order_count'] > mean_d + 2 * std_d]
ax.scatter(spikes['date'], spikes['order_count'],
           color='red', zorder=5, s=40, label='Demand Spike')

ax.set_title('Daily Order Demand — Dec 2023 to Nov 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Orders')
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('plot_01_daily_demand.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Spikes detected: {len(spikes)} days above mean+2sigma ({mean_d:.1f} + 2x{std_d:.1f})')


In [ ]:
# -- 4.2  Heatmap: Hour x Day of Week ------------------------------
hour_dow = (
    completed
    .groupby(['day_of_week', 'hour'])
    .size()
    .unstack('hour')
    .fillna(0)
)
hour_dow.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    hour_dow, cmap='YlOrRd', linewidths=0.3,
    cbar_kws={'label': 'Order Count'},
    annot=False, ax=ax
)
ax.set_title('Demand Heatmap — Hour of Day x Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day (0-23)')
ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.savefig('plot_02_heatmap_hour_dow.png', dpi=150, bbox_inches='tight')
plt.show()

peak_hour = hour_dow.sum().idxmax()
peak_day  = hour_dow.sum(axis=1).idxmax()
print(f'Peak hour: {peak_hour}:00 | Peak day: {peak_day}')


In [ ]:
# -- 4.3  Monthly demand — seasonal bar chart ----------------------
monthly = (
    completed
    .groupby(['year', 'month'])
    .agg(order_count=('order_id','count'), revenue=('total_revenue','sum'))
    .reset_index()
)
monthly['period'] = pd.to_datetime(monthly[['year','month']].assign(day=1))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

unique_years = sorted(monthly['year'].unique())
year_color_map = {y: c for y, c in zip(unique_years, ['#90CAF9', '#1565C0', '#42A5F5', '#0D47A1'])}
colors = [year_color_map[y] for y in monthly['year']]

ax1.bar(monthly['period'], monthly['order_count'], width=20, color=colors, alpha=0.85)
ax1.set_ylabel('Order Count')
ax1.set_title('Monthly Order Volume & Revenue', fontsize=14, fontweight='bold')

ax2.bar(monthly['period'], monthly['revenue'] / 1e6, width=20, color=colors, alpha=0.85)
ax2.set_ylabel('Revenue (Rp Juta)')
ax2.set_xlabel('Month')
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=year_color_map[y], label=str(y)) for y in unique_years]
ax1.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig('plot_03_monthly_demand.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -- 4.4  Revenue by Top 5 Category — stacked monthly --------------
TOP_CATS = (
    completed['product_categories']
    .value_counts()
    .head(5)
    .index
    .tolist()
)
print('Top 5 categories (by order count):', TOP_CATS)

cat_monthly = (
    completed[completed['product_categories'].isin(TOP_CATS)]
    .assign(period=lambda x: pd.to_datetime(x[['year','month']].assign(day=1)))
    .groupby(['period','product_categories'])['total_revenue']
    .sum()
    .unstack('product_categories')
    .fillna(0)
) / 1e6   # convert to Rp Juta

fig, ax = plt.subplots(figsize=(14, 5))
cat_monthly.plot(kind='bar', stacked=True, ax=ax,
                 colormap='tab10', width=0.8, alpha=0.85)
ax.set_title('Monthly Revenue by Top 5 Product Category (Rp Juta)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (Rp Juta)')
ax.set_xticklabels([str(p.date())[:7] for p in cat_monthly.index], rotation=45, ha='right')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('plot_04_category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -- 4.5  Stationarity Test (ADF) — wajib sebelum ARIMA ------------
if HAS_STATSMODELS:
    series = daily_ts['order_count'].values
    adf_result = adfuller(series, autolag='AIC')

    print('=== Augmented Dickey-Fuller Test ===')
    print(f'ADF Statistic : {adf_result[0]:.4f}')
    print(f'p-value       : {adf_result[1]:.4f}')
    print(f'Critical Values: {adf_result[4]}')

    if adf_result[1] < 0.05:
        print('\n✅ Series is STATIONARY — ARIMA can proceed without differencing')
    else:
        print('\n⚠️  Series is NON-STATIONARY — will apply d=1 differencing in ARIMA')
else:
    series = daily_ts['order_count'].values
    print('⏭️  Skip: statsmodels tidak terinstall.')


In [ ]:
# -- 4.6  ACF & PACF Plot — menentukan p, q untuk ARIMA ------------
if HAS_STATSMODELS:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(series,  lags=40, ax=ax1, title='Autocorrelation (ACF)')
    plot_pacf(series, lags=40, ax=ax2, title='Partial Autocorrelation (PACF)')
    plt.suptitle('ACF & PACF — Identifying ARIMA Parameters (p, q)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plot_05_acf_pacf.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Interpretation: significant lags in PACF -> suggest p value for AR term')
else:
    print('⏭️  Skip: statsmodels tidak terinstall.')


---
## 5. Train / Test Split

- **Train:** Dec 2023 – ~22 bulan pertama
- **Test:**  60 hari terakhir (out-of-sample evaluation)

Menggunakan 60 hari terakhir sebagai test set memberikan holdout yang realistis untuk ketiga model.


In [ ]:
TEST_DAYS  = 60
train_ts   = daily_ts.iloc[:-TEST_DAYS].copy()
test_ts    = daily_ts.iloc[-TEST_DAYS:].copy()

print(f'Train: {len(train_ts)} days  ({train_ts["date"].min().date()} -> {train_ts["date"].max().date()})')
print(f'Test : {len(test_ts)} days  ({test_ts["date"].min().date()} -> {test_ts["date"].max().date()})')

# Helper: evaluation metrics
def evaluate(y_true, y_pred, model_name='Model'):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    print(f'{model_name:20s}  MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%')
    return {'model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
y_train = train_ts['order_count'].values
y_test  = test_ts['order_count'].values


---
## 6. Model 1 — ARIMA (Baseline)

**ARIMA(p,d,q)** — classical statistical baseline
- p (AR order): dari plot PACF
- d (differencing): 1 jika non-stationary, 0 jika stationary
- q (MA order): dari plot ACF

Expected weakness: tidak bisa handle multiple seasonality (daily + weekly)


In [ ]:
arima_preds = None

if HAS_STATSMODELS:
    # ARIMA(7,1,2) — p=7 (weekly lag), d=1 (1 differencing), q=2 (MA smoothing)
    # Sesuaikan p,d,q berdasarkan output ACF/PACF di atas kalau perlu
    arima_model = ARIMA(y_train, order=(7, 1, 2))
    arima_fit   = arima_model.fit()
    print(arima_fit.summary())
else:
    print('⏭️  Skip: statsmodels tidak terinstall.')


In [ ]:
if HAS_STATSMODELS:
    # Walk-forward forecast on test set
    arima_preds = []
    history     = list(y_train)

    for t in range(len(y_test)):
        model = ARIMA(history, order=(7, 1, 2))
        fit   = model.fit()
        yhat  = fit.forecast(steps=1)[0]
        arima_preds.append(max(0, yhat))   # clip negative forecasts
        history.append(y_test[t])          # update history dengan nilai aktual
        if (t + 1) % 10 == 0:
            print(f'  ...walk-forward step {t + 1}/{len(y_test)}')

    arima_preds = np.array(arima_preds)
    results.append(evaluate(y_test, arima_preds, 'ARIMA(7,1,2)'))
else:
    print('⏭️  Skip: statsmodels tidak terinstall.')


---
## 7. Model 2 — Facebook Prophet (Main Model)

**Prophet** dipilih sebagai main model karena:
- Native handle **multiple seasonality** (weekly + yearly)
- Bisa inject **holiday regressors** — penting untuk Shopee (Harbolnas, Lebaran)
- Interpretable: bisa lihat komponen trend vs seasonality
- Robust terhadap missing data & outlier


In [ ]:
prophet_preds = None

if HAS_PROPHET:
    # ------------------------------------------------------------------
    # Siapkan dataframe format Prophet: ds (date), y (target)
    # ------------------------------------------------------------------
    prophet_train = train_ts[['date','order_count','holiday_flag','promo_flag']].copy()
    prophet_train.columns = ['ds','y','holiday_flag','promo_flag']

    prophet_test = test_ts[['date','order_count','holiday_flag','promo_flag']].copy()
    prophet_test.columns = ['ds','y','holiday_flag','promo_flag']

    # ------------------------------------------------------------------
    # Dataframe holiday Indonesia untuk Prophet
    # ------------------------------------------------------------------
    id_holidays = pd.DataFrame({
        'holiday': [
            'Tahun Baru','Imlek','Imlek','Imlek',
            'Isra Miraj','Wafat Isa Almasih','Idul Fitri','Idul Fitri',
            'Idul Fitri','Idul Fitri','Idul Fitri','Hari Buruh',
            'Kenaikan Isa Almasih','Waisak','Pancasila','Idul Adha',
            'Idul Adha','Tahun Baru Islam','Kemerdekaan','Maulid Nabi',
            'Natal','Natal',
            # 2025
            'Tahun Baru','Imlek','Imlek','Imlek',
            'Idul Fitri','Idul Fitri','Idul Fitri','Idul Fitri',
            'Idul Fitri','Wafat Isa Almasih','Hari Buruh','Kenaikan Isa Almasih',
            'Waisak','Pancasila','Idul Adha','Kemerdekaan',
            'Maulid Nabi','Natal'
        ],
        'ds': pd.to_datetime([
            '2024-01-01','2024-02-08','2024-02-09','2024-02-10',
            '2024-03-11','2024-03-29','2024-04-08','2024-04-09',
            '2024-04-10','2024-04-11','2024-04-12','2024-05-01',
            '2024-05-09','2024-05-23','2024-06-01','2024-06-17',
            '2024-06-18','2024-07-07','2024-08-17','2024-09-16',
            '2024-12-25','2024-12-26',
            '2025-01-01','2025-01-27','2025-01-28','2025-01-29',
            '2025-03-28','2025-03-29','2025-03-30','2025-03-31',
            '2025-04-01','2025-04-18','2025-05-01','2025-05-12',
            '2025-05-29','2025-06-01','2025-06-06','2025-08-17',
            '2025-09-05','2025-12-25'
        ]),
        'lower_window': 0,
        'upper_window': 1
    })

    print(f'Holiday dataframe: {len(id_holidays)} entries')
else:
    print('⏭️  Skip: prophet tidak terinstall.')


In [ ]:
if HAS_PROPHET:
    prophet_model = Prophet(
        holidays=id_holidays,
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.1,
        seasonality_prior_scale=10,
        holidays_prior_scale=5,
        interval_width=0.95
    )

    prophet_model.add_regressor('promo_flag')
    prophet_model.add_regressor('holiday_flag')

    prophet_model.fit(prophet_train)
    print('✅ Prophet model fitted')
else:
    print('⏭️  Skip: prophet tidak terinstall.')


In [ ]:
if HAS_PROPHET:
    prophet_preds_raw = prophet_model.predict(prophet_test[['ds','promo_flag','holiday_flag']])
    prophet_preds = np.maximum(0, prophet_preds_raw['yhat'].values)

    results.append(evaluate(y_test, prophet_preds, 'Prophet'))

    fig = prophet_model.plot_components(prophet_preds_raw)
    plt.suptitle('Prophet Model Components — Trend, Weekly, Yearly Seasonality',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('plot_06_prophet_components.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⏭️  Skip: prophet tidak terinstall.')


---
## 8. Model 3 — LSTM (Advanced)

**LSTM (Long Short-Term Memory)** — deep learning untuk pola nonlinear
- Menangkap dependency jangka panjang yang tidak bisa ditangkap ARIMA/Prophet
- Input: sequence 30 hari sebelumnya -> prediksi 1 hari ke depan
- Features: order_count + fitur tambahan (holiday, promo, cyclical date encoding)

Tradeoff: less interpretable, butuh lebih banyak data, training lebih lama


In [ ]:
lstm_preds = None

if HAS_TF:
    LOOKBACK = 30   # gunakan 30 hari terakhir untuk prediksi 1 hari ke depan

    # ------------------------------------------------------------------
    # 8.1  Siapkan fitur multi-dimensi untuk LSTM
    # ------------------------------------------------------------------
    lstm_features = ['order_count','holiday_flag','promo_flag']

    daily_ts['dow_sin']   = np.sin(2 * np.pi * daily_ts['date'].dt.dayofweek / 7)
    daily_ts['dow_cos']   = np.cos(2 * np.pi * daily_ts['date'].dt.dayofweek / 7)
    daily_ts['month_sin'] = np.sin(2 * np.pi * daily_ts['date'].dt.month / 12)
    daily_ts['month_cos'] = np.cos(2 * np.pi * daily_ts['date'].dt.month / 12)

    lstm_features += ['dow_sin','dow_cos','month_sin','month_cos']
    n_features = len(lstm_features)

    # ------------------------------------------------------------------
    # 8.2  Fit scaler HANYA pada train (hindari data leakage dari test set)
    # ------------------------------------------------------------------
    n_train_rows = len(daily_ts) - TEST_DAYS

    scaler = MinMaxScaler()
    scaler.fit(daily_ts[lstm_features].values[:n_train_rows])
    scaled_data = scaler.transform(daily_ts[lstm_features].values)

    scaled_target_scaler = MinMaxScaler()
    scaled_target_scaler.fit(daily_ts[['order_count']].values[:n_train_rows])
    scaled_y = scaled_target_scaler.transform(daily_ts[['order_count']].values)

    # ------------------------------------------------------------------
    # 8.3  Sliding window sequences  [X: (lookback, n_feat)] -> [y: 1]
    # ------------------------------------------------------------------
    def create_sequences(data, target, lookback):
        X, y = [], []
        for i in range(lookback, len(data)):
            X.append(data[i-lookback:i])
            y.append(target[i])
        return np.array(X), np.array(y)

    X_all, y_all = create_sequences(scaled_data, scaled_y, LOOKBACK)

    # Selaraskan train/test split dengan TEST_DAYS
    split_idx = len(X_all) - TEST_DAYS
    X_train, X_test = X_all[:split_idx], X_all[split_idx:]
    y_train_lstm, y_test_lstm = y_all[:split_idx], y_all[split_idx:]

    print(f'X_train shape: {X_train.shape}  y_train shape: {y_train_lstm.shape}')
    print(f'X_test  shape: {X_test.shape}   y_test  shape: {y_test_lstm.shape}')
else:
    print('⏭️  Skip: tensorflow tidak terinstall.')


In [ ]:
if HAS_TF:
    tf.random.set_seed(42)

    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOKBACK, n_features)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ], name='DemandForecast_LSTM')

    lstm_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='huber'
    )

    lstm_model.summary()
else:
    print('⏭️  Skip: tensorflow tidak terinstall.')


In [ ]:
if HAS_TF:
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-5)
    ]

    history_lstm = lstm_model.fit(
        X_train, y_train_lstm,
        epochs=100,
        batch_size=16,
        validation_split=0.15,
        callbacks=callbacks,
        verbose=1
    )

    print(f'\nTraining stopped at epoch: {len(history_lstm.history["loss"])}')
else:
    print('⏭️  Skip: tensorflow tidak terinstall.')


In [ ]:
if HAS_TF:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history_lstm.history['loss'],     label='Train Loss', color='#1565C0')
    ax.plot(history_lstm.history['val_loss'], label='Val Loss',   color='#EF5350')
    ax.set_title('LSTM Training & Validation Loss', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Huber Loss')
    ax.legend()
    plt.tight_layout()
    plt.savefig('plot_07_lstm_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⏭️  Skip: tensorflow tidak terinstall.')


In [ ]:
if HAS_TF:
    lstm_preds_scaled = lstm_model.predict(X_test)
    lstm_preds = scaled_target_scaler.inverse_transform(lstm_preds_scaled).flatten()
    lstm_preds = np.maximum(0, lstm_preds)

    results.append(evaluate(y_test, lstm_preds, 'LSTM'))
else:
    print('⏭️  Skip: tensorflow tidak terinstall.')


---
## 9. Model Evaluation & Comparison

In [ ]:
# ------------------------------------------------------------------
# 9.1  Tabel perbandingan metrik
# ------------------------------------------------------------------
if len(results) == 0:
    raise RuntimeError(
        'Tidak ada model yang berhasil dilatih (statsmodels/prophet/tensorflow semuanya '
        'tidak terinstall). Install minimal salah satu lalu jalankan ulang dari Section 6-8.'
    )

results_df = pd.DataFrame(results).set_index('model')

# Improvement dihitung relatif terhadap model pertama yang berhasil jalan (baseline)
baseline_name = results_df.index[0]
baseline_mae  = results_df.loc[baseline_name, 'MAE']
results_df['MAE_improvement_%'] = (
    (baseline_mae - results_df['MAE']) / baseline_mae * 100
).round(1)

print('=' * 65)
print(f'{"MODEL":<22} {"MAE":>8} {"RMSE":>8} {"MAPE":>8} {"vs " + baseline_name:>14}')
print('=' * 65)
for model, row in results_df.iterrows():
    imp = f"+{row['MAE_improvement_%']:.1f}%" if row['MAE_improvement_%'] > 0 else f"{row['MAE_improvement_%']:.1f}%"
    print(f'{model:<22} {row["MAE"]:>8.2f} {row["RMSE"]:>8.2f} {row["MAPE"]:>7.2f}% {imp:>14}')
print('=' * 65)

best_model = results_df['MAE'].idxmin()
print(f'\n🏆 Best model by MAE: {best_model}')


In [ ]:
# ------------------------------------------------------------------
# 9.2  Visual comparison — Actual vs model yang tersedia
# ------------------------------------------------------------------
test_dates = pd.to_datetime(test_ts['date'].values)

candidate_preds = [
    ('ARIMA(7,1,2)', arima_preds,   '#FF7043'),
    ('Prophet',      prophet_preds, '#1565C0'),
    ('LSTM',         lstm_preds,    '#2E7D32'),
]
model_preds = [(n, p, c) for n, p, c in candidate_preds if p is not None]

fig, axes = plt.subplots(len(model_preds), 1, figsize=(14, 4 * len(model_preds)), sharex=True)
if len(model_preds) == 1:
    axes = [axes]

for ax, (name, preds, color) in zip(axes, model_preds):
    ax.plot(test_dates, y_test, color='#333333', linewidth=2, label='Actual', zorder=3)
    ax.plot(test_dates, preds, color=color, linewidth=1.8,
            linestyle='--', label=f'{name} Forecast', alpha=0.9)
    ax.fill_between(test_dates, y_test, preds, alpha=0.12, color=color)
    mae_val  = mean_absolute_error(y_test, preds)
    mape_val = np.mean(np.abs((y_test - preds) / np.where(y_test == 0, 1, y_test))) * 100
    ax.set_title(f'{name}  |  MAE={mae_val:.2f}  MAPE={mape_val:.2f}%',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Orders')
    ax.legend(loc='upper left')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Date')
plt.suptitle('Model Comparison: Actual vs Predicted Demand (60-Day Test Set)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_08_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ------------------------------------------------------------------
# 9.3  Metrics bar chart — MAE, RMSE, MAPE berdampingan
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['MAE', 'RMSE', 'MAPE']
bar_colors = ['#FF7043', '#1565C0', '#2E7D32', '#8E24AA'][:len(results_df)]

for ax, metric in zip(axes, metrics):
    vals   = results_df[metric].values
    models = results_df.index.tolist()
    bars   = ax.bar(models, vals, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=1.5)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01*max(vals),
                f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric + (' (%)' if metric == 'MAPE' else ''))
    ax.set_ylim(0, max(vals) * 1.25)
    ax.tick_params(axis='x', rotation=10)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Model Performance Metrics Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_09_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 10. Business Impact Calculation

Menerjemahkan akurasi model menjadi estimasi penghematan biaya operasional.
Semua angka asumsi sekarang **dihitung langsung dari dataset**, bukan angka hardcode,
sehingga otomatis menyesuaikan kalau datanya berubah.


In [ ]:
# ------------------------------------------------------------------
# Asumsi bisnis — dihitung langsung dari data, bukan hardcode
# ------------------------------------------------------------------
CANCELLED_ORDER_RATE  = (df['order_status'] == 'Cancelled').mean()
AVG_REVENUE_PER_ORDER = completed['total_revenue'].mean()
AVG_SHIPPING_COST     = completed['estimated_shipping_cost'].mean()
TOTAL_MONTHLY_ORDERS  = daily_ts['order_count'].mean() * 30

# Baseline error = model pertama (baseline); best = model dengan MAPE terkecil
PLANNING_ERROR_RATE_BEFORE = results_df.loc[baseline_name, 'MAPE'] / 100
PLANNING_ERROR_RATE_AFTER  = results_df['MAPE'].min() / 100

best_model_name = results_df['MAE'].idxmin()
mae_improvement = results_df.loc[baseline_name, 'MAE'] - results_df.loc[best_model_name, 'MAE']
mape_reduction  = results_df.loc[baseline_name, 'MAPE'] - results_df.loc[best_model_name, 'MAPE']

wasted_dispatch_before  = TOTAL_MONTHLY_ORDERS * PLANNING_ERROR_RATE_BEFORE * AVG_SHIPPING_COST
wasted_dispatch_after   = TOTAL_MONTHLY_ORDERS * PLANNING_ERROR_RATE_AFTER  * AVG_SHIPPING_COST
monthly_dispatch_saving = wasted_dispatch_before - wasted_dispatch_after

print('=' * 60)
print('BUSINESS IMPACT SUMMARY')
print('=' * 60)
print(f'Cancelled order rate (data)   : {CANCELLED_ORDER_RATE*100:.2f}%')
print(f'Avg revenue / order (data)    : Rp {AVG_REVENUE_PER_ORDER:,.0f}')
print(f'Avg shipping cost (data)      : Rp {AVG_SHIPPING_COST:,.0f}')
print(f'Avg monthly order volume      : {TOTAL_MONTHLY_ORDERS:,.0f} orders')
print('---')
print(f'Best model vs baseline  : {best_model_name} vs {baseline_name}')
print(f'MAE improvement         : {mae_improvement:.2f} orders/day')
print(f'MAPE reduction          : {mape_reduction:.2f}%')
print('---')
print(f'Est. planning error before : {PLANNING_ERROR_RATE_BEFORE*100:.1f}% MAPE')
print(f'Est. planning error after  : {PLANNING_ERROR_RATE_AFTER*100:.1f}% MAPE')
print('---')
print(f'Monthly wasted dispatch (before) : Rp {wasted_dispatch_before:>14,.0f}')
print(f'Monthly wasted dispatch (after)  : Rp {wasted_dispatch_after:>14,.0f}')
print(f'📦 Estimated monthly saving      : Rp {monthly_dispatch_saving:>14,.0f}')
print(f'📦 Estimated annual  saving      : Rp {monthly_dispatch_saving*12:>14,.0f}')
print('=' * 60)


In [ ]:
# ------------------------------------------------------------------
# Simpan tabel hasil akhir
# ------------------------------------------------------------------
results_df.to_csv('model_evaluation_results.csv')
print('✅ Results saved to model_evaluation_results.csv')

print('\n=== Final Model Evaluation Table ===')
print(results_df.to_string())


---
## Summary

| Step | What was done |
|---|---|
| **Cleaning** | Normalisasi status, parsing datetime, konsolidasi 45+ varian pengiriman |
| **Feature Eng** | 18 fitur: temporal, cyclical encoding, holiday/promo flag, COD flag |
| **EDA** | Heatmap, plot seasonality, ACF/PACF, stationarity test |
| **ARIMA** | Baseline — statistik klasik, menangani autokorelasi linear |
| **Prophet** | Main model — holiday regressors, seasonality multiplicative |
| **LSTM** | Advanced — pola nonlinear, 30-hari lookback window (scaler tanpa leakage) |
| **Evaluation** | Perbandingan MAE / RMSE / MAPE + estimasi business impact (dihitung dari data) |

**Catatan robustness:** seluruh bagian modeling (ARIMA/Prophet/LSTM) dibungkus pengecekan
`HAS_STATSMODELS` / `HAS_PROPHET` / `HAS_TF` sehingga notebook tetap bisa dijalankan
end-to-end walau salah satu library belum terinstall — bagian itu otomatis di-skip
dengan pesan yang jelas, bukan error yang menghentikan seluruh notebook.
